In [79]:
# ═══════════════════════════════════════════════════════════════════════════
# Section 10 — Full Environment Setup
# This cell is self-contained: run it after all previous notebook cells OR
# as a standalone starting point. It replicates every constant, path,
# helper and ROM-loading step needed by the inversion section.
# ═══════════════════════════════════════════════════════════════════════════

import json, pickle, warnings, time
from pathlib import Path

import h5py
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import NullLocator
from matplotlib.lines import Line2D
from scipy.interpolate import RBFInterpolator
from scipy.stats import norm as sp_norm

warnings.filterwarnings("ignore")

# ── rcParams (identical to parent notebook) ──────────────────────────────
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],
    'text.latex.preamble': r'\usepackage{amsmath} \usepackage{amssymb} \usepackage{bm}',
    'font.size': 12, 'axes.labelsize': 12, 'axes.titlesize': 12,
    'legend.fontsize': 12, 'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': False, 'ytick.minor.visible': False,
    'axes.linewidth': 1, 'lines.linewidth': 1.5, 'lines.markersize': 6,
    'axes.grid': False, 'grid.alpha': 0.20,
    'figure.dpi': 150, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
    'legend.frameon': True, 'legend.framealpha': 1.0,
    'legend.edgecolor': 'black', 'legend.fancybox': False,
    'legend.handlelength': 1.5, 'legend.labelspacing': 0.4,
    'legend.handletextpad': 0.5, 'legend.borderaxespad': 0.5,
    'xtick.major.pad': 5.0, 'ytick.major.pad': 5.0,
})

# ── Colour palette ────────────────────────────────────────────────────────
C1, C2, C3, C4  = "#1a5e9e", "#0d6e56", "#9b2a1a", "#b07318"
C_MRST, C_ROM   = "#2C5F8A", "#C0392B"
C_DUH           = "#6A0572"
C_INV           = "#1a7a4a"   # recovered -- green
C_GUESS         = "#b07318"   # initial guess -- amber
C_TRUE          = "#2C5F8A"   # true parameter marker -- matches C_MRST

# ── Directory layout (matches parent notebook exactly) ────────────────────
# Adjust these paths if your working directory differs from the notebook root
_HERE        = Path(".").resolve()
TRAINING_DIR = _HERE / "../data/01_training"
VAL_DIR      = _HERE / "../data/02_validation"
ROM_DIR      = _HERE / "../rom_vrate"
FIG_DIR      = _HERE / "../figures_vrate"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def sf(name):
    plt.savefig(FIG_DIR / name, dpi=600, bbox_inches='tight')
    print(f"  Saved: {FIG_DIR / name}")

# ── Reservoir constants (field units) ────────────────────────────────────
P_INIT  = 4500.0        # psia
NX, NY  = 50, 50
N_CELLS = NX * NY
N_STEPS = 350
T_START, T_END = 0.001, 5000.0
PHI    = 0.18
CT     = 1.2e-5         # psi^-1
MU_CP  = 1.2            # cp
BO     = 1.15           # RB/STB
H_FT   = 150.0          # ft
RW_FT  = 0.35           # ft
C_WBS  = 0.003          # bbl/psi
DX_FT  = 30 * 3.28084   # ft
RE_FT  = (NX * 30 / 2) * 3.28084
WC_IDX = (NY//2 - 1)*NX + (NX//2 - 1)   # 1224 (0-indexed)

K_GRID = [10, 20, 50, 100, 200, 300]
S_GRID = [0, 1, 2, 4, 6, 8]
SCHEDS = list(range(1, 8))

# ── I/O helpers ───────────────────────────────────────────────────────────
def load_mat(filepath, key):
    try:
        return sio.loadmat(str(filepath), variable_names=[key])[key]
    except NotImplementedError:
        with h5py.File(str(filepath), "r") as f:
            arr = f[key][()]
            return arr.T if arr.ndim == 2 else arr

def load_well_data(case_dir):
    wd  = case_dir / "well_data.mat"
    bhp = load_mat(wd, "bhp_psia").ravel()
    t   = load_mat(wd, "time_hr").ravel()
    q   = load_mat(wd, "rate_STBd").ravel()
    dp  = load_mat(wd, "delta_p_psi").ravel()
    with open(case_dir / "params.json") as f:
        pm = json.load(f)
    return bhp, t, q, dp, pm

def get_schedule_shape(pm):
    rates = np.atleast_1d(pm["rates_STBd"])
    t_ch  = np.atleast_1d(pm.get("t_change_hr", []))
    return rates, t_ch

def physics_screen(k, s):
    kh    = k * H_FT
    t_wbs = 170000.0 * C_WBS * MU_CP * np.exp(0.14*s) / kh
    t_pss = (PHI * MU_CP * CT * RE_FT**2) / (0.000264 * k) * 0.1
    if t_wbs >= T_END:
        return False, t_wbs, t_pss, 0, 0, 0, "t_WBS >= T_end"
    if t_wbs >= t_pss:
        return False, t_wbs, t_pss, 0, 0, 0, "no MTR"
    return True, t_wbs, t_pss, 0, 0, 0, "VALID"

VALID_PAIRS = [(k, s) for k in K_GRID for s in S_GRID
               if physics_screen(k, s)[0]]

# ── Bourdet derivative ────────────────────────────────────────────────────
def bourdet(t_hr, dp, L=0.2):
    ln_t = np.log(t_hr)
    d = np.full_like(dp, np.nan)
    for i in range(1, len(t_hr)-1):
        il = i-1
        while il > 0  and (ln_t[i]-ln_t[il]) < L: il -= 1
        ir = i+1
        while ir < len(t_hr)-1 and (ln_t[ir]-ln_t[i]) < L: ir += 1
        dl = ln_t[i]-ln_t[il]; dr = ln_t[ir]-ln_t[i]
        if dl > 0 and dr > 0:
            d[i] = ((dp[i]-dp[il])/dl*dr + (dp[ir]-dp[i])/dr*dl)/(dl+dr)
    m = np.isfinite(d) & (d > 0) & (dp > 0)
    return t_hr[m], dp[m], d[m]

# ── Load pre-built ROM components ─────────────────────────────────────────
print("Loading ROM from:", ROM_DIR.resolve())
Vn     = np.load(ROM_DIR / "basis" / "Vn.npy")
N_MODES = Vn.shape[1]
idir   = ROM_DIR / "interpolants"

def _load_interp(nm):
    with open(idir / f"{nm}_interp.pkl", "rb") as f:
        obj = pickle.load(f)
    return obj["i"], obj["s"]

A_interps, A_shape = _load_interp("A")
B_interps, B_shape = _load_interp("B")
C_interps, C_shape = _load_interp("C")
D_interps, D_shape = _load_interp("D")

# Reconstruct normalisation parameters from the training operator points
ops_dir      = ROM_DIR / "operators"
results_meta = []
for k, s in VALID_PAIRS:
    tag = f"k{k:03d}_s{s:02d}"
    if (ops_dir / tag / "A_hat.npy").exists():
        results_meta.append({"tag": tag, "k": k, "S": s})

pts_lst = [[np.log10(r["k"]), r["S"]] for r in results_meta]
pts     = np.array(pts_lst)
pts_min = pts.min(axis=0)
pts_max = pts.max(axis=0)
pts_rng = pts_max - pts_min
pts_rng[pts_rng == 0] = 1.0

def eval_op(interps, shape, k_q, s_q):
    pt   = np.array([[np.log10(k_q), float(s_q)]])
    pt_n = (pt - pts_min) / pts_rng
    return np.array([f(pt_n)[0] for f in interps]).reshape(shape)

# ── Core ROM integrator (Backward Euler, matches parent notebook) ─────────
def rom_bhp(k_star, s_star, t_hr, q_vec):
    A = eval_op(A_interps, A_shape, k_star, s_star)
    B = eval_op(B_interps, B_shape, k_star, s_star)
    C = eval_op(C_interps, C_shape, k_star, s_star)
    D = eval_op(D_interps, D_shape, k_star, s_star)
    K   = len(t_hr)
    dt  = np.diff(t_hr, prepend=0.0); dt[0] = t_hr[0]
    In  = np.eye(N_MODES)
    xk  = np.zeros(N_MODES)
    Yr  = np.zeros(K)
    for j in range(K):
        rhs = xk + dt[j] * B.ravel() * q_vec[j]
        xk  = np.linalg.solve(In - dt[j]*A, rhs)
        Yr[j] = float(C @ xk + D * q_vec[j])
    return P_INIT + Yr

# ── Sanity check ──────────────────────────────────────────────────────────
print(f"  ROM loaded: {N_MODES} modes, {len(results_meta)} training points")
print(f"  Valid (k,S) pairs : {len(VALID_PAIRS)}")
print(f"  Training DIR      : {TRAINING_DIR.resolve()}")
print(f"  Validation DIR    : {VAL_DIR.resolve()}")
print(f"  ROM DIR           : {ROM_DIR.resolve()}")

# Quick smoke-test: evaluate ROM at a training point
_k0, _s0 = VALID_PAIRS[0]
_cd = TRAINING_DIR / f"k{_k0:03d}_s{_s0:02d}_sched1"
if (_cd / "well_data.mat").exists():
    _bhp, _t, _q, _, _ = load_well_data(_cd)
    _pred = rom_bhp(_k0, _s0, _t, _q)
    _err  = np.linalg.norm(_pred - _bhp) / (np.linalg.norm(P_INIT - _bhp) + 1e-9)
    print(f"\n  Smoke-test at (k={_k0}, S={_s0}): relative BHP error = {_err:.2e}")
    if _err < 0.05:
        print("  Environment OK -- ready for Section 10.")
    else:
        print("  WARNING: error > 5% -- check ROM paths or data integrity.")
else:
    print("  Smoke-test skipped (training data not found at expected path).")

Loading ROM from: F:\Niloy Deb\Part Two\reservoir_rom\rom_vrate
  ROM loaded: 15 modes, 36 training points
  Valid (k,S) pairs : 36
  Training DIR      : F:\Niloy Deb\Part Two\reservoir_rom\data\01_training
  Validation DIR    : F:\Niloy Deb\Part Two\reservoir_rom\data\02_validation
  ROM DIR           : F:\Niloy Deb\Part Two\reservoir_rom\rom_vrate

  Smoke-test at (k=10, S=0): relative BHP error = 3.73e-03
  Environment OK -- ready for Section 10.


In [80]:
# ── 10.1  RBF gradient (central FD in normalised parameter space) ────────
def rbf_grad_op(interps, shape, k_q, s_q, eps=1e-5):
    xi_n = np.array([[(np.log10(k_q) - pts_min[0]) / pts_rng[0],
                       (float(s_q)   - pts_min[1]) / pts_rng[1]]])
    n_elem = int(np.prod(shape))
    g_lk = np.zeros(n_elem)
    g_s  = np.zeros(n_elem)
    for c, rbf in enumerate(interps):
        xf_lk = xi_n.copy(); xf_lk[0,0] += eps
        xb_lk = xi_n.copy(); xb_lk[0,0] -= eps
        xf_s  = xi_n.copy(); xf_s[0,1]  += eps
        xb_s  = xi_n.copy(); xb_s[0,1]  -= eps
        g_lk[c] = (rbf(xf_lk)[0] - rbf(xb_lk)[0]) / (2*eps)
        g_s[c]  = (rbf(xf_s)[0]  - rbf(xb_s)[0])  / (2*eps)
    # Chain-rule: normalised -> physical coordinates
    g_lk /= pts_rng[0]
    g_s  /= pts_rng[1]
    return g_lk.reshape(shape), g_s.reshape(shape)

# ── FIX 1: Stability enforcement and consistent shifted gradients ──────
def _enforce_stability(A, margin=1e-6):
    max_re = np.max(np.real(np.linalg.eigvals(A)))
    if max_re >= 0:
        A = A - (max_re + margin) * np.eye(A.shape[0])
    return A

def get_stable_A(k_q, s_q):
    A = eval_op(A_interps, A_shape, k_q, s_q)
    return _enforce_stability(A)

def fd_grad_stable_A(k_star, s_star, eps=1e-5):
    dk = pts_rng[0] * eps
    ds = pts_rng[1] * eps
    
    A_fw_k = get_stable_A(10**(np.log10(k_star) + dk), s_star)
    A_bw_k = get_stable_A(10**(np.log10(k_star) - dk), s_star)
    dA_lk = (A_fw_k - A_bw_k) / (2 * eps * pts_rng[0])
    
    A_fw_s = get_stable_A(k_star, s_star + ds)
    A_bw_s = get_stable_A(k_star, s_star - ds)
    dA_s = (A_fw_s - A_bw_s) / (2 * eps * pts_rng[1])
    
    return dA_lk, dA_s

# ── 10.2  Backward-Euler forward pass + sensitivity propagation ──────────
def rom_bhp_sensitivity(k_star, s_star, t_hr, q_vec):
    A = eval_op(A_interps, A_shape, k_star, s_star)
    B = eval_op(B_interps, B_shape, k_star, s_star)
    C = eval_op(C_interps, C_shape, k_star, s_star)
    D = eval_op(D_interps, D_shape, k_star, s_star)

    # Enforce stability on A
    A = _enforce_stability(A)

    # Use numerical gradients for the shifted A matrix
    dA_lk, dA_s = fd_grad_stable_A(k_star, s_star)
    
    # B, C, D are un-shifted, so analytical RBF gradients are still exact
    dB_lk, dB_s = rbf_grad_op(B_interps, B_shape, k_star, s_star)
    dC_lk, dC_s = rbf_grad_op(C_interps, C_shape, k_star, s_star)
    dD_lk, dD_s = rbf_grad_op(D_interps, D_shape, k_star, s_star)

    K   = len(t_hr)
    dt  = np.diff(t_hr, prepend=0.0); dt[0] = t_hr[0]
    In  = np.eye(N_MODES)
    xk  = np.zeros(N_MODES)
    Plk = np.zeros(N_MODES)
    Ps  = np.zeros(N_MODES)

    bhp = np.zeros(K); dlk = np.zeros(K); ds = np.zeros(K)

    C_v    = C.ravel()
    dC_lkv = dC_lk.ravel()
    dC_sv  = dC_s.ravel()
    d_lk   = float(dD_lk.ravel()[0])
    d_s    = float(dD_s.ravel()[0])
    d_val  = float(D.ravel()[0])

    for j in range(K):
        h   = dt[j]; u = q_vec[j]
        Mi  = np.linalg.solve(In - h*A, In)

        # State update
        xk  = Mi @ (xk + h * B.ravel() * u)

        # Sensitivity update
        Plk = Mi @ (Plk + h * dA_lk @ xk + h * dB_lk.ravel() * u)
        Ps  = Mi @ (Ps  + h * dA_s  @ xk + h * dB_s.ravel()  * u)

        bhp[j] = P_INIT + float(C_v @ xk) + d_val * u
        dlk[j] = float(dC_lkv @ xk) + float(C_v @ Plk) + d_lk * u
        ds[j]  = float(dC_sv  @ xk) + float(C_v @ Ps)  + d_s  * u

    return bhp, dlk, ds

def rom_bhp(k_star, s_star, t_hr, q_vec):
    A = eval_op(A_interps, A_shape, k_star, s_star)
    B = eval_op(B_interps, B_shape, k_star, s_star)
    C = eval_op(C_interps, C_shape, k_star, s_star)
    D = eval_op(D_interps, D_shape, k_star, s_star)
    A = _enforce_stability(A)
    K   = len(t_hr)
    dt  = np.diff(t_hr, prepend=0.0); dt[0] = t_hr[0]
    In  = np.eye(N_MODES)
    xk  = np.zeros(N_MODES)
    Yr  = np.zeros(K)
    C_v = C.ravel()
    d_v = float(D.ravel()[0])
    for j in range(K):
        rhs = xk + dt[j] * B.ravel() * q_vec[j]
        xk  = np.linalg.solve(In - dt[j]*A, rhs)
        Yr[j] = float(C_v @ xk) + d_v * q_vec[j]
    return P_INIT + Yr

In [81]:
# ── 10.3  Verify analytical gradient against central finite-difference ────
k_test, s_test, sid_gv = 150.0, 2.0, 1  # S=2 in S_GRID; sched 1 always exists
cd_gv = None
for base in [VAL_DIR, TRAINING_DIR]:
    p = base / f"k{int(k_test):03d}_s{int(s_test):02d}_sched{sid_gv}"
    if (p / "well_data.mat").exists(): cd_gv = p; break

if cd_gv:
    _, t_gv, q_gv, _, _ = load_well_data(cd_gv)
    bhp_an, dlk_an, ds_an = rom_bhp_sensitivity(k_test, s_test, t_gv, q_gv)

    eps = 1e-3
    bfwd_lk, _, _ = rom_bhp_sensitivity(k_test*10**eps,  s_test,     t_gv, q_gv)
    bbwd_lk, _, _ = rom_bhp_sensitivity(k_test*10**(-eps),s_test,    t_gv, q_gv)
    bfwd_s,  _, _ = rom_bhp_sensitivity(k_test,           s_test+eps, t_gv, q_gv)
    bbwd_s,  _, _ = rom_bhp_sensitivity(k_test,           s_test-eps, t_gv, q_gv)
    dlk_fd = (bfwd_lk - bbwd_lk) / (2*eps)
    ds_fd  = (bfwd_s  - bbwd_s)  / (2*eps)

    err_lk = np.max(np.abs(dlk_an - dlk_fd)) / (np.max(np.abs(dlk_fd)) + 1e-9)
    err_s  = np.max(np.abs(ds_an  - ds_fd))  / (np.max(np.abs(ds_fd))  + 1e-9)

    print(f"Gradient verification at k={k_test} mD, S={s_test}")
    print(f"  Max relative error  d/d(log10_k) : {err_lk:.2e}")
    print(f"  Max relative error  d/dS          : {err_s:.2e}")
    print("  PASSED" if err_lk < 0.02 and err_s < 0.02 else "  CHECK REQUIRED")
else:
    print("Test data not found -- verification skipped.")

Gradient verification at k=150.0 mD, S=2.0
  Max relative error  d/d(log10_k) : 2.11e-06
  Max relative error  d/dS          : 9.79e-09
  PASSED


In [82]:
# ── 10.4  Levenberg-Marquardt inversion with parameter scaling ──────────
def lm_invert(t_obs, p_obs, q_obs, xi0, bounds,
              max_iter=100, tol=1e-8, lam0=1e-2, nu=5.0, verbose=True):
    (lk_lo, lk_hi), (S_lo, S_hi) = bounds
    lk_rng = lk_hi - lk_lo          # physical range of log10_k
    S_rng  = S_hi  - S_lo           # physical range of S
    scale  = np.array([lk_rng, S_rng])   # xi_physical = xi_normalised * scale

    K   = len(t_obs)
    dt  = np.diff(t_obs, prepend=0.0); dt[0] = t_obs[0]
    w   = np.sqrt(dt); w /= (w.mean() + 1e-12)

    xi  = np.array(xi0, dtype=float)   # physical (log10_k, S)
    lam = lam0
    xi_hist, J_hist, grad_hist, step_hist = [xi.copy()], [], [], []

    clamp = lambda v: np.array([np.clip(v[0], lk_lo, lk_hi),
                                  np.clip(v[1], S_lo,  S_hi)])

    def mj(xi_):
        k_, s_ = 10**xi_[0], xi_[1]
        b_, dl_, ds_ = rom_bhp_sensitivity(k_, s_, t_obs, q_obs)
        r_  = w * (b_ - p_obs)
        # Scale Jacobian columns to normalised space:
        Jac = np.column_stack([w * dl_ * scale[0],
                                w * ds_ * scale[1]])
        return r_, Jac, 0.5 * np.dot(r_, r_)

    r, J, Jv = mj(xi); J_hist.append(Jv)
    if verbose:
        h = (f"{'It':>3}  {'log10k':>7}  {'S':>6}  {'k[mD]':>9}  "
             f"{'J':>11}  {'||grad||':>10}  {'lam':>8}")
        print(h); print('-' * len(h))

    converged = False
    for it in range(max_iter):
        JtJ = J.T @ J
        Jtg = J.T @ r
        gn  = np.linalg.norm(Jtg)
        grad_hist.append(gn)

        # LM solve in normalised space
        Hlm     = JtJ + lam * np.diag(np.diag(JtJ) + 1e-12)
        delta_n = np.linalg.solve(Hlm, -Jtg)       # step in normalised space
        delta_p = delta_n * scale                   # convert to physical space
        step_hist.append(np.linalg.norm(delta_p))

        xi_new = clamp(xi + delta_p)
        rn, Jn, Jnv = mj(xi_new)

        if verbose:
            print(f"{it+1:>3}  {xi[0]:>7.4f}  {xi[1]:>6.3f}  {10**xi[0]:>9.2f}"
                  f"  {Jv:>11.4e}  {gn:>10.4e}  {lam:>8.3e}")

        if Jnv < Jv:
            xi, r, J, Jv = xi_new, rn, Jn, Jnv
            lam = max(lam / nu, 1e-12)
        else:
            lam = min(lam * nu, 1e8)

        # FIX 2: Break if step size drops below tolerance to prevent infinite rejections
        if np.linalg.norm(delta_p) < 1e-8:
            converged = True
            if verbose:
                print(f"\n  Converged (step size {np.linalg.norm(delta_p):.2e} < tol).")
            break

        xi_hist.append(xi.copy()); J_hist.append(Jv)
        
        # Original gradient norm check
        if gn < tol:
            converged = True
            if verbose:
                print(f"\n  Converged at iteration {it+1} (||grad||={gn:.2e}).")
            break

    if verbose and not converged:
        print(f"  Max iterations ({max_iter}) reached.")

    kf, sf = 10**xi[0], xi[1]
    sig2 = Jv / max(K - 2, 1)
    try:
        inv_n = np.linalg.inv(J.T @ J)
        cov   = sig2 * (inv_n * np.outer(scale, scale))
        ci_lk = 1.96 * np.sqrt(cov[0, 0])
        ci_s  = 1.96 * np.sqrt(cov[1, 1])
        cor   = cov[0, 1] / (np.sqrt(cov[0, 0] * cov[1, 1]) + 1e-12)
    except Exception:
        cov = np.full((2, 2), np.nan); ci_lk = ci_s = cor = np.nan

    return xi_hist, J_hist, xi, dict(
        xi_hist=xi_hist, J_hist=J_hist, grad_hist=grad_hist,
        step_hist=step_hist, converged=converged,
        k_final=kf, s_final=sf, ci95_k_log=ci_lk, ci95_s=ci_s,
        corr_ks=cor, cov_xi=cov, sigma2=sig2,
        residuals=r / (w + 1e-12))

In [83]:
# ── 10.5  Load blind test case ────────────────────────────────────────────
K_TRUE, S_TRUE, SID_INV = 150, 5, 4

cd_blind = None
for base in [VAL_DIR, TRAINING_DIR]:
    p = base / f"k{K_TRUE:03d}_s{S_TRUE:02d}_sched{SID_INV}"
    if (p / "well_data.mat").exists(): cd_blind = p; break

if cd_blind is None:
    raise FileNotFoundError(
        f"MRST data missing: k={K_TRUE}, S={S_TRUE}, sched={SID_INV}")

bhp_obs, t_obs, q_obs, _, pm_blind = load_well_data(cd_blind)
rates_blind, t_ch_blind = get_schedule_shape(pm_blind)
print(f"Blind case loaded: k={K_TRUE} mD, S={S_TRUE}, Schedule {SID_INV}")
print(f"Rate: {' -> '.join(str(int(r)) for r in rates_blind)} STB/d"
      f"  |  t_change = {t_ch_blind} hr")
print(f"Observations: {len(t_obs)} time steps  "
      f"t in [{t_obs[0]:.3f}, {t_obs[-1]:.0f}] hr")

bhp_fwd_check = rom_bhp(K_TRUE, S_TRUE, t_obs, q_obs)
fwd_err = (np.linalg.norm(bhp_fwd_check - bhp_obs) /
           (np.linalg.norm(P_INIT - bhp_obs) + 1e-9))
print(f"\nROM forward error at true (k={K_TRUE}, S={S_TRUE}): {fwd_err:.2e}")
print("  OK -- proceed with inversion." if fwd_err < 0.05
      else "  WARNING: forward error > 5% -- inversion result may be biased.")

Blind case loaded: k=150 mD, S=5, Schedule 4
Rate: 1000 -> 500 STB/d  |  t_change = [1000] hr
Observations: 350 time steps  t in [0.001, 5000] hr

ROM forward error at true (k=150, S=5): 3.54e-01


In [84]:
# ── 10.6  Misfit landscape scan (40x35 ROM evaluations) ─────────────────
NK, NS = 40, 35
k_vec  = np.logspace(np.log10(5), np.log10(300), NK)
s_vec  = np.linspace(0, 8, NS)
kk, ss = np.meshgrid(k_vec, s_vec)
JJ     = np.full_like(kk, np.nan)

dt_ls = np.diff(t_obs, prepend=0.0); dt_ls[0] = t_obs[0]
w_ls  = np.sqrt(dt_ls); w_ls /= (w_ls.mean() + 1e-12)

print(f"Scanning {NK}x{NS}={NK*NS} points ...")
t0 = time.perf_counter()
for i, si in enumerate(s_vec):
    for j, kj in enumerate(k_vec):
        try:
            bh  = rom_bhp(kj, si, t_obs, q_obs)
            r_  = w_ls * (bh - bhp_obs)
            JJ[i,j] = 0.5*np.dot(r_,r_)
        except Exception:
            pass
elapsed = time.perf_counter() - t0
print(f"Done -- {elapsed:.1f} s  ({elapsed/(NK*NS)*1e3:.1f} ms/call)")

Scanning 40x35=1400 points ...
Done -- 5.4 s  (3.8 ms/call)


In [85]:
# ── 10.7  Run LM inversion ────────────────────────────────────────────────
# FIX 3: Warm-start using the global minimum from the misfit landscape scan (Cell 10.6)
min_idx = np.nanargmin(JJ)
K_INIT = kk.ravel()[min_idx]
S_INIT = ss.ravel()[min_idx]

xi0    = (np.log10(K_INIT), S_INIT)
bounds = ((np.log10(5), np.log10(300)), (0.0, 8.0))

print(f"True:    k = {K_TRUE} mD,  S = {S_TRUE}")
print(f"Initial (Warm-Start): k = {K_INIT:.2f} mD,  S = {S_INIT:.2f}\n")

t0_inv = time.perf_counter()
xi_hist, J_hist, xi_final, diag = lm_invert(
    t_obs, bhp_obs, q_obs, xi0=xi0, bounds=bounds,
    max_iter=100, tol=1e-8, lam0=1e-2, nu=5.0, verbose=True)
t_inv = time.perf_counter() - t0_inv

K_EST = diag['k_final']; S_EST = diag['s_final']
ci_klo = 10**(xi_final[0] - diag['ci95_k_log'])
ci_khi = 10**(xi_final[0] + diag['ci95_k_log'])

print(f"\n  Recovered:  k = {K_EST:.2f} mD  (err {abs(K_EST-K_TRUE)/K_TRUE*100:.1f}%),  "
      f"S = {S_EST:.3f}  (err {abs(S_EST-S_TRUE):.3f})")
print(f"  95% CI:     k in [{ci_klo:.1f}, {ci_khi:.1f}] mD;  "
      f"S = {S_EST:.2f} +/- {diag['ci95_s']:.3f}")
print(f"  Corr(log_k, S) = {diag['corr_ks']:.3f}")
print(f"  Wall time: {t_inv:.2f} s  |  Converged: {diag['converged']}")

True:    k = 150 mD,  S = 5
Initial (Warm-Start): k = 143.87 mD,  S = 0.94

 It   log10k       S      k[mD]            J    ||grad||       lam
------------------------------------------------------------------
  1   2.1580   0.941     143.87   4.9166e+02  1.6229e+05  1.000e-02
  2   2.1649   0.976     146.18   2.8542e+02  4.3522e+03  2.000e-03
  3   2.1649   0.976     146.18   2.8542e+02  4.3522e+03  1.000e-02
  4   2.1659   0.990     146.53   2.8506e+02  4.0807e+02  2.000e-03
  5   2.1659   0.990     146.53   2.8506e+02  4.0807e+02  1.000e-02
  6   2.1660   0.992     146.56   2.8506e+02  2.8733e+01  2.000e-03
  7   2.1660   0.992     146.56   2.8506e+02  2.8733e+01  1.000e-02
  8   2.1660   0.992     146.57   2.8506e+02  4.0052e+00  2.000e-03
  9   2.1660   0.992     146.57   2.8506e+02  4.0052e+00  1.000e-02
 10   2.1660   0.992     146.57   2.8506e+02  5.0863e-01  2.000e-03
 11   2.1660   0.992     146.57   2.8506e+02  5.0863e-01  1.000e-02
 12   2.1660   0.992     146.57   2.8506e+